# 소비재 분야 기업들의 감사보수에 영향을 주는 요인 분석

소비재(식품·음료·생활용품·화장품) 분야 상장기업 12개사의 재무데이터와 감사보수 데이터를 읽어들여
데이터를 통합·분석 및 전처리한 후, 다양한 머신러닝 회귀 모델로 감사보수를 예측하고
어떤 요인이 감사보수에 영향을 주는지 분석합니다.

---
company: 기업명 / year: 회계연도 / industry: 업종 <br>
assets: 자산총계 / revenue: 매출액 / debt: 부채총계 / equity: 자본총계 <br>
auditor: 감사인(회계법인명) / audit_opinion: 감사의견 <br>
audit_fee: 감사보수(백만원) / audit_hours: 감사시간 <br>

---
> 데이터 출처: DART(전자공시시스템) Open API로 사전 수집한 데이터입니다.
> 실제 API로 데이터를 수집하고 싶다면 별도 제공된 `dart_audit_fee_collector.py`를 참고하세요.


## 0. 전 과정에서 사용할 기본 라이브러리 준비

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
plt.rc("font", family="NanumGothicCoding")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

import random
import tensorflow as tf
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)


## 1. 데이터 수집 및 통합

재무정보와 감사보수정보가 각각 다른 CSV 파일로 되어 있어, 두 데이터를 기업명(company)과
회계연도(year) 기준으로 병합하여 하나의 데이터프레임으로 통합합니다.

In [ ]:
fin_df = pd.read_csv("소비재_재무정보.csv")
audit_df = pd.read_csv("소비재_감사보수정보.csv")

print(fin_df.shape, audit_df.shape)

df = pd.merge(fin_df, audit_df, on=["company", "year"], how="inner")
print(df.shape)
df.head()


In [ ]:
df.info()


## 2. 범주형 변수 EDA (분포 파악)

감사인(auditor), 업종(industry), 감사의견(audit_opinion)의 분포를 확인합니다.

In [ ]:
sns.countplot(data=df, x="auditor")
plt.xticks(rotation=45)
plt.title("감사인(회계법인)별 데이터 개수")
plt.show()


In [ ]:
sns.countplot(data=df, x="audit_opinion")
plt.title("감사의견 분포")
plt.show()


In [ ]:
sns.boxplot(data=df, x="industry", y="audit_fee")
plt.title("업종별 감사보수 분포")
plt.show()


## 3. 범주형 변수 이진화 (피처 엔지니어링)

감사보수 결정요인 관련 선행연구에서 자주 다루는 두 가지 이진 변수를 생성합니다.
- `is_big4`: 감사인이 Big4(삼일·삼정·한영·안진회계법인) 여부
- `is_clean_opinion`: 감사의견이 '적정'인지 여부

In [ ]:
big4_list = ["삼일회계법인", "삼정회계법인", "한영회계법인", "안진회계법인"]

df["is_big4"] = df["auditor"].isin(big4_list).astype(int)
df["is_clean_opinion"] = (df["audit_opinion"] == "적정").astype(int)

df[["auditor", "is_big4", "audit_opinion", "is_clean_opinion"]].drop_duplicates().head(10)


In [ ]:
sns.barplot(data=df, x="is_big4", y="audit_fee")
plt.title("Big4 여부에 따른 평균 감사보수")
plt.xlabel("Big4 여부 (0: 비Big4, 1: Big4)")
plt.show()


## 4. 다른 변수들 간의 상관관계 분석

In [ ]:
plt.figure(figsize=(9, 7))
sns.heatmap(df.corr(numeric_only=True), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("변수 간 상관관계")
plt.show()


위 히트맵을 보면 `assets`(자산총계), `debt`(부채총계), `equity`(자본총계), `revenue`(매출액)
사이의 상관계수가 매우 높습니다(0.79~0.98). 이는 우연이 아니라 **회계항등식**
(자산총계 = 부채총계 + 자본총계) 때문으로, `equity`는 `assets`와 `debt`만으로 완전히 결정되는
중복 정보입니다. 이 상태로 모델링하면 다중공선성 때문에 회귀계수가 불안정해지고, 트리 모델의
변수 중요도도 여러 변수에 쪼개져 특정 변수(예: assets)의 실제 영향력이 과소평가될 수 있습니다.

In [ ]:
print(df[["assets", "debt", "equity", "revenue"]].corr())


## 5. 불필요 변수 제거

기업명(company)과 회계연도(year)는 식별자 성격의 변수이고, auditor·audit_opinion은
이미 이진 변수(is_big4, is_clean_opinion)로 피처 엔지니어링을 완료했으므로 원본 컬럼을 제거합니다.

또한 회계항등식으로 인한 다중공선성을 해소하기 위해 `equity`는 완전히 제거하고,
`assets`는 감사보수 결정요인 선행연구에서 흔히 쓰이는 로그 규모 지표 `log_assets`로,
`debt`는 절대금액 대신 `leverage`(부채비율 = 부채총계/자산총계)로 변환합니다.

In [ ]:
df["log_assets"] = np.log(df["assets"])
df["leverage"] = df["debt"] / df["assets"]

df_reduced = df.drop(
    ["company", "year", "auditor", "audit_opinion", "assets", "debt", "equity"], axis=1
)
df_reduced.head()


## 6. 결측치 처리

In [ ]:
print(df_reduced.isnull().sum())

df_clean = df_reduced.dropna().reset_index(drop=True)
df_clean.shape


## 7. 범주형 변수의 수치화 (인코딩)

남아있는 범주형 변수는 업종(industry) 하나이며, get_dummies로 원-핫 인코딩합니다.

In [ ]:
df_encoded = pd.get_dummies(df_clean, columns=["industry"])
df_encoded.head()


## 8. 데이터 분할 및 정규화

감사보수(audit_fee)를 예측 대상(y)으로, 나머지 변수를 피처(X)로 설정하여
훈련/검증 데이터셋을 8:2로 분할하고 StandardScaler로 정규화합니다.

In [ ]:
X = df_encoded.drop("audit_fee", axis=1)
y = df_encoded["audit_fee"]

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)

print(X_train_scaled.shape, X_valid_scaled.shape)


## 9. 다양한 머신러닝 회귀 모델 학습

선형회귀, 랜덤포레스트, 그래디언트부스팅, XGBoost 네 가지 회귀 모델을 학습합니다.

In [ ]:
lr = LinearRegression()
rf = RandomForestRegressor(n_estimators=200, max_depth=5, random_state=42)
gbr = GradientBoostingRegressor(n_estimators=200, max_depth=3, learning_rate=0.05, random_state=42)
xgbr = XGBRegressor(n_estimators=200, max_depth=3, learning_rate=0.05, random_state=42)

models = {"LinearRegression": lr, "RandomForest": rf, "GradientBoosting": gbr, "XGBoost": xgbr}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    print(f"{name} 학습 완료")


## 10. 모델 성능 평가 및 최적 모델 선정

MAE, RMSE, R2 세 가지 지표로 네 개 모델의 성능을 비교하고 최적 모델을 선정합니다.

In [ ]:
results = []

for name, model in models.items():
    pred = model.predict(X_valid_scaled)
    mae = mean_absolute_error(y_valid, pred)
    rmse = mean_squared_error(y_valid, pred) ** 0.5
    r2 = r2_score(y_valid, pred)
    results.append({"model": name, "MAE": mae, "RMSE": rmse, "R2": r2})

result_df = pd.DataFrame(results).sort_values("R2", ascending=False).reset_index(drop=True)
result_df


In [ ]:
best_model_name = result_df.iloc[0]["model"]
best_model = models[best_model_name]
print("최적 모델:", best_model_name)

if hasattr(best_model, "feature_importances_"):
    importance = pd.Series(best_model.feature_importances_, index=X.columns).sort_values(ascending=False)
    sns.barplot(x=importance.values, y=importance.index)
    plt.title(f"Feature Importance ({best_model_name})")
    plt.show()
    print(importance)


## 11. 학습 곡선 시각화를 통한 모델 학습 상태 점검

딥러닝(Keras) 모델을 추가로 학습하여, epoch별 train loss와 validation loss 추이를 통해
모델이 과소적합/과대적합 없이 잘 학습되었는지 점검합니다.

In [ ]:
model_dl = Sequential([
    Dense(64, activation="relu", input_shape=(X_train_scaled.shape[1],)),
    Dense(128, activation="relu"),
    BatchNormalization(),
    Dropout(0.2),
    Dense(64, activation="relu"),
    Dense(1, activation="linear"),
])

model_dl.compile(optimizer="adam", loss="mse", metrics=["mae"])

es = EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)
mc = ModelCheckpoint("best_audit_fee_model.h5", save_best_only=True)

history = model_dl.fit(
    X_train_scaled, y_train,
    epochs=100, batch_size=8,
    validation_data=(X_valid_scaled, y_valid),
    callbacks=[es, mc],
    verbose=0,
)

print("학습 종료, 총 epoch:", len(history.history["loss"]))


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history.history["loss"], label="train loss")
plt.plot(history.history["val_loss"], label="validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss (MSE)")
plt.title("학습 곡선 (Learning Curve)")
plt.legend()
plt.show()


In [ ]:
dl_pred = model_dl.predict(X_valid_scaled).flatten()
dl_mae = mean_absolute_error(y_valid, dl_pred)
dl_r2 = r2_score(y_valid, dl_pred)

print("딥러닝 모델 MAE:", dl_mae)
print("딥러닝 모델 R2:", dl_r2)


### 분석 결론

- 상관관계 분석과 Feature Importance 결과, **Big4 여부(is_big4)가 감사보수를 결정하는
  압도적으로 지배적인 요인**(상관계수 0.83, RandomForest 중요도 약 69%)으로 나타났습니다.
- **자산 규모(log_assets)는 중간 수준의 양(+)의 상관관계(0.52)**를 보이며 감사보수에
  유의미하게 기여하지만, 이번 표본에서는 Big4 여부에 비해 상대적으로 작은 비중
  (RandomForest 중요도 약 12%)을 차지했습니다.
- 이는 "기업 규모와 감사인 유형이 감사보수의 핵심 결정요인"이라는 기존 연구의 방향성과는
  일치하지만, 두 요인의 **상대적 크기**는 본 표본에서 Big4 여부 쪽으로 크게 치우쳐 있다는
  점에 유의해야 합니다. 두 변수의 영향력을 동등한 수준으로 서술하는 것은 과장된 해석입니다.
- 상관관계 분석 과정에서 assets·debt·equity·revenue 간의 높은 상관관계(0.79~0.98)를
  발견했고, 이는 회계항등식(자산=부채+자본)에 따른 다중공선성임을 확인했습니다. equity를
  제거하고 assets→log_assets, debt→leverage(부채비율)로 변환하여 이 문제를 해소했습니다.
- 학습 곡선을 통해 딥러닝 모델이 특정 시점 이후 validation loss가 더 이상 개선되지 않아
  EarlyStopping으로 적절히 종료되었음을 확인했습니다.
- 다만 표본 수(12개사 x 6개년 = 72건)가 적어 일반화에는 한계가 있고, 감사보수 데이터 자체가
  실제 DART 응답 필드를 확정하지 못해 가상으로 생성한 샘플이므로, Big4 프리미엄과 규모 효과의
  상대적 비율은 실제 데이터로 교체 시 달라질 수 있습니다. 본 분석은 감사보수 결정요인을
  데이터 기반으로 탐색하고, 그 결과를 정확하게 해석하는 접근법을 보여주기 위한 포트폴리오입니다.
